# FetchPickAndPlace: factorised SAC/TBTRL across five goals

Uses the current SAC trainer, factorised actor and twin critics, actor replay/representation penalties, carried entropy temperature, and fixed global replay budget. **Dense Fetch rewards; no HER.** Scratch, sequential transfer and independent recovery use the same five fixed targets in metres.

Install `uv sync --locked --group dev --extra notebooks --extra robotics` (or `pip install -e ".[notebooks,robotics]"`) and restart the kernel first. Dependencies pin a tested Gymnasium-Robotics/MuJoCo combination. On a Linux machine without a display, select `MUJOCO_GL=egl` **before importing MuJoCo**, or run the kernel under Xvfb with GLFW.

The physical reset uses `initial_state_seed=42` on **every** episode, independently of the learning/evaluation seed and target. This fixes the robot and, for PickAndPlace, the block. Goals are set internally before stepping so rewards, success and the rendered target agree. Fetch keeps its native continuing-task behaviour and ends at the 50-step time limit; success does not terminate early.

The same learning, loss, representation, gradient, retention and cost figures appear inline. A labelled 640×480 simulator rollout GIF replaces maze Q/arrow maps and path plots after each trained goal. One deterministic GIF is enough for a fixed-start policy; `gif_episodes` can request more. Numeric evaluation still uses eight episodes, which are identical for a deterministic policy at this fixed start—not independent random-start trials.

In [ ]:
import os
from pathlib import Path
from uuid import uuid4

import numpy as np
import torch

from tbtrl.environments.registry import make_environment
from tbtrl.experiments.config import load_config
from tbtrl.experiments.diagnostics import NotebookDiagnostics
from tbtrl.experiments.plots import plot_comparison, plot_run
from tbtrl.experiments.runner import run_experiment

root = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "configs/tbtrl_fetch_pick_and_place_sac.json").exists()
)
SMOKE = False
DEVICE = (
    "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
)
MODES = ("scratch", "sequential", "recovery")
config = load_config(root / "configs/tbtrl_fetch_pick_and_place_sac.json")
if SMOKE:
    config = config.smoke()
config.validate()
output_root = Path(os.environ.get("TBTRL_OUTPUT_ROOT", str(root / "runs")))
output = output_root / f"{config.name}-{uuid4().hex[:8]}"
print("SMOKE CHECK" if SMOKE else "FULL RESEARCH RUN")
print(f"Environment={config.environment}; device={DEVICE}; training seeds={config.seeds}")
print(
    f"Fixed physical reset seed={config.environment_options['initial_state_seed']}; dense rewards; no HER"
)
print("Ordered target positions (metres):", list(enumerate(config.goals)))
print(
    f"Steps/goal={config.training['total_steps']:,}; warmup={config.training['warmup_steps']:,}; recovery={config.recovery['total_steps']:,}"
)
print(
    f"Global replay slots={config.global_replay_capacity:,}; recent slots={config.recent_replay_capacity:,}"
)
get_ipython().run_line_magic("matplotlib", "inline")

In [ ]:
# Confirm physical starts match after motion, across seeds and across target goals.
reference = None
for task_id, goal in enumerate(config.goals):
    env = make_environment(config.environment, goal, **config.environment_options)
    try:
        initial, _ = env.reset(seed=123 + task_id)
        env.step(np.array([0.4, -0.3, 0.2, -1.0], dtype=np.float32))
        repeated, _ = env.reset(seed=987 + task_id)
        np.testing.assert_array_equal(initial, repeated)
        if reference is None:
            reference = initial.copy()
        np.testing.assert_array_equal(reference, initial)
        print(
            f"Goal {task_id}: fixed start verified; gripper={initial[:3]}; achieved={initial[env.achieved_slice]}"
        )
    finally:
        env.close()

In [ ]:
diagnostics = NotebookDiagnostics(rollout_episodes=config.eval_episodes, gif_episodes=1)
records = run_experiment(config, output, device=DEVICE, modes=MODES, on_task_end=diagnostics)
print("Saved run:", output)

In [ ]:
for row in records:
    memory = row["replay_memory"]
    print(
        f"{row['mode']:10s} goal={row['task_id']} steps={row['steps']:,} "
        f"archive={memory['archive_samples']:,}/{memory['archive_capacity']:,} "
        f"allocated={memory['allocated_transition_bytes'] / 1_000_000:.2f} MB"
    )
figure = plot_run(output)
figure

In [ ]:
comparison = plot_comparison(output)
comparison

Full runs use five goals; `SMOKE=True` uses two goals and small budgets only to verify execution. The trainer/update equations are shared with the maze experiments. Fetch configs explicitly use 256-wide networks and unclipped Bellman targets; rewards, observation/action dimensions and time limits follow Fetch.

Inspect retention across all five goals and independent recovery costs before drawing conclusions. Dense rewards do not guarantee that PickAndPlace will learn to grasp; these notebooks establish the experiment, not a claimed successful robotics result. The earlier sandbox used SB3 SAC with HER, which is a different method.

Per-goal directories contain the usual metrics/figures plus `rollout-01.gif`, a final PNG frame and `rollouts.json` with initial state, goal, return, success and final distance. `render_saved_run(output, display=True)` regenerates figures and GIFs from saved checkpoints without retraining. Set `spatial=False` to omit simulator rendering. See `docs/fetch-experiments.md` for details.